In [ ]:
import getpass
import os

os.environ["GOOGLE_API_KEY"] = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)

In [ ]:
from google import genai
from google.genai import types
import pathlib
import json

# ✅ Configure Gemini
# genai.configure(api_key="YOUR_GEMINI_API_KEY")  # Uncomment and add key

# ✅ Path to your local PDF
filepath = pathlib.Path("/content/FTP_Chapters/FTP2023_Chapter01.pdf")

# ✅ Prompt for structured chunking
prompt = """
Break this policy document into structured sections.

Each section must have:
- "section_title": the heading of the section
- "section_content": all text under that heading

Return a valid JSON array like:
[
  { "section_title": "...", "section_content": "..." },
  ...
]
"""

# ✅ Call Gemini
client = genai.Client()
response = client.models.generate_content(
    model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
    contents=[
        types.Part.from_bytes(
            data=filepath.read_bytes(),
            mime_type='application/pdf',
        ),
        prompt
    ]
)

# ✅ Clean the response
raw = response.text.strip()
for wrapper in ("```json", "```"):
    if raw.startswith(wrapper):
        raw = raw[len(wrapper):]
    if raw.endswith("```"):
        raw = raw[:-3]
raw = raw.strip()

# ✅ Parse and save to JSON
try:
    chunks = json.loads(raw)
    print(f"✅ Extracted {len(chunks)} chunks\n")

    for i, section in enumerate(chunks[:3]):  # Preview
        print(f"🔹 Section {i+1}: {section['section_title']}")
        print(section["section_content"][:300], "...\n")

    # ✅ Save the chunks as JSON
    output_path = "/content/FTP_Chapters/FTP2023_Chapter01.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)
    print(f"💾 JSON saved to: {output_path}")

except Exception as e:
    print("❌ Failed to parse response:")
    print(raw[:500])


✅ Extracted 36 chunks

🔹 Section 1: Chapter 1 Legal Framework and Trade Facilitation
Legal Framework and Trade Facilitation ...

🔹 Section 2: A. LEGAL FRAMEWORK
 ...

🔹 Section 3: 1.00 Legal Basis of Foreign Trade Policy
The Foreign Trade Policy (FTP) 2023 is notified by Central Government, in exercise of powers conferred under Section 5 of the Foreign Trade (Development & Regulation) Act, 1992 (No. 22 of 1992) [FT (D&R) Act], as amended. ...

💾 JSON saved to: /content/FTP_Chapters/FTP2023_Chapter01.json


In [ ]:
from google import genai
from google.genai import types
import pathlib
import json
import os

# ✅ Configure Gemini
# genai.configure(api_key="YOUR_GEMINI_API_KEY")  # Uncomment and add your key

# ✅ Set input and output folders
pdf_folder = pathlib.Path("/content/FTP_Chapters")
output_folder = pdf_folder  # Save JSONs in same folder

# ✅ Prompt for structured chunking
prompt = """
Break this policy document into structured sections.

Each section must have:
- "section_title": the heading of the section
- "section_content": all text under that heading

Return a valid JSON array like:
[
  { "section_title": "...", "section_content": "..." },
  ...
]
"""

# ✅ Initialize Gemini client
client = genai.Client()

# ✅ Function to process a single file
def process_pdf(pdf_path: pathlib.Path):
    print(f"\n📄 Processing: {pdf_path.name}")
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(
                data=pdf_path.read_bytes(),
                mime_type='application/pdf',
            ),
            prompt
        ]
    )

    # Clean response
    raw = response.text.strip()
    for wrapper in ("```json", "```"):
        if raw.startswith(wrapper):
            raw = raw[len(wrapper):]
        if raw.endswith("```"):
            raw = raw[:-3]
    raw = raw.strip()

    # Try parsing and saving
    try:
        chunks = json.loads(raw)
        print(f"✅ Extracted {len(chunks)} sections")

        # Save JSON
        output_path = output_folder / (pdf_path.stem + ".json")
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(chunks, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to: {output_path.name}")
    except Exception as e:
        print("❌ Failed to parse JSON:")
        print(raw[:500])

# ✅ Iterate over all PDFs in the folder
for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    process_pdf(pdf_file)



📄 Processing: Chapter 6 FTP 2023.pdf
✅ Extracted 24 sections
💾 Saved to: Chapter 6 FTP 2023.json

📄 Processing: FTP2023_Chapter01.pdf
✅ Extracted 35 sections
💾 Saved to: FTP2023_Chapter01.json

📄 Processing: FTP2023_Chapter03.pdf
✅ Extracted 8 sections
💾 Saved to: FTP2023_Chapter03.json

📄 Processing: FTP2023_Chapter04 as on 13.02.2025.pdf
✅ Extracted 69 sections
💾 Saved to: FTP2023_Chapter04 as on 13.02.2025.json

📄 Processing: FTP2023_Chapter05.pdf
✅ Extracted 14 sections
💾 Saved to: FTP2023_Chapter05.json

📄 Processing: FTP2023_Chapter07.pdf
✅ Extracted 12 sections
💾 Saved to: FTP2023_Chapter07.json

📄 Processing: FTP2023_Chapter08.pdf
✅ Extracted 10 sections
💾 Saved to: FTP2023_Chapter08.json

📄 Processing: FTP2023_Chapter09.pdf
✅ Extracted 16 sections
💾 Saved to: FTP2023_Chapter09.json

📄 Processing: FTP2023_Chapter10.pdf
✅ Extracted 13 sections
💾 Saved to: FTP2023_Chapter10.json

📄 Processing: FTP2023_Chapter11.pdf
✅ Extracted 63 sections
💾 Saved to: FTP2023_Chapter11.json

📄 Pr

In [ ]:
from google import genai
from google.genai import types
import pathlib
import json
import os

# ✅ Configure Gemini
# genai.configure(api_key="YOUR_GEMINI_API_KEY")  # Uncomment and add your key

# ✅ Set input and output folders
pdf_folder = pathlib.Path("/content/FTP_Chapters")
output_folder = pdf_folder  # Save JSONs in same folder

# ✅ Prompt for paragraph-based chunking (longer, semantically rich)
prompt = """
Divide this policy document into large semantic paragraphs.

Each chunk should represent a full idea or topic (about 200–400 words).

Return as a JSON array where each entry contains:
- "section_title": inferred or extracted heading (if applicable)
- "section_content": the full paragraph text

Output:
[
  { "section_title": "...", "section_content": "..." },
  ...
]
"""

# ✅ Initialize Gemini client
client = genai.Client()

# ✅ Function to process a single file
def process_pdf(pdf_path: pathlib.Path):
    print(f"\n📄 Processing: {pdf_path.name}")
    response = client.models.generate_content(
        model=(__import__('sys').path.insert(0, '..') or __import__('config').LLM_MODEL),
        contents=[
            types.Part.from_bytes(
                data=pdf_path.read_bytes(),
                mime_type='application/pdf',
            ),
            prompt
        ]
    )

    # Clean response
    raw = response.text.strip()
    for wrapper in ("```json", "```"):
        if raw.startswith(wrapper):
            raw = raw[len(wrapper):]
        if raw.endswith("```"):
            raw = raw[:-3]
    raw = raw.strip()

    # Try parsing and saving
    try:
        chunks = json.loads(raw)
        print(f"✅ Extracted {len(chunks)} semantic chunks")

        # Save JSON
        output_path = output_folder / (pdf_path.stem + ".json")
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(chunks, f, indent=2, ensure_ascii=False)
        print(f"💾 Saved to: {output_path.name}")
    except Exception as e:
        print("❌ Failed to parse JSON:")
        print(raw[:500])

# ✅ Iterate over all PDFs in the folder
for pdf_file in sorted(pdf_folder.glob("*.pdf")):
    process_pdf(pdf_file)



📄 Processing: FTP2023_Chapter01.pdf
✅ Extracted 15 semantic chunks
💾 Saved to: FTP2023_Chapter01.json

📄 Processing: FTP2023_Chapter03.pdf
✅ Extracted 5 semantic chunks
💾 Saved to: FTP2023_Chapter03.json

📄 Processing: [UPDATED] CHAPTER 2 OF FTP.pdf
✅ Extracted 32 semantic chunks
💾 Saved to: [UPDATED] CHAPTER 2 OF FTP.json


In [ ]:
import json
import pathlib

# ✅ Set the folder containing the individual JSON files
json_folder = pathlib.Path("/content/FTP_Chapters")

# ✅ Output file name
output_path = json_folder / "FTP2023_AllChapters.json"

# ✅ List to hold all chunks from all files
all_chunks = []

# ✅ Loop over each .json file
for json_file in sorted(json_folder.glob("*.json")):
    if json_file.name.endswith(".json") and not json_file.name.endswith("AllChapters.json"):
        print(f"📦 Merging: {json_file.name}")
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list):
                all_chunks.extend(data)
            else:
                print(f"⚠️ Skipped {json_file.name} (not a list)")

# ✅ Save concatenated chunks
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\n✅ All JSONs merged into: {output_path}")
print(f"📊 Total chunks: {len(all_chunks)}")


📦 Merging: FTP2023_Chapter01.json
📦 Merging: FTP2023_Chapter03.json
📦 Merging: [UPDATED] CHAPTER 2 OF FTP.json

✅ All JSONs merged into: /content/FTP_Chapters/FTP2023_AllChapters.json
📊 Total chunks: 52


In [ ]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_postgres.vectorstores import PGVector

# === CONFIGURATION ===
GEMINI_API_KEY = (__import__('sys').path.insert(0, '..') or __import__('config').GOOGLE_API_KEY)
PGVECTOR_CONN_STRING = (__import__('sys').path.insert(0, '..') or __import__('config').DB_CONNECTION_STRING)
JSON_PATH = "/content/FTP_Chapters/FTP2023_AllChapters.json"
COLLECTION_NAME = "VectorDB"

# === STEP 1: Load JSON & Convert to Document Objects ===
def load_documents(json_path):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    docs = []
    for chunk in data:
        title = chunk.get("section_title")
        text = chunk.get("section_content")
        if title and text:
            docs.append(Document(page_content=text, metadata={"section_title": title}))
    return docs

documents = load_documents(JSON_PATH)
print(f"✅ Loaded {len(documents)} document chunks.")

# === STEP 2: Setup Embedding Model ===
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/embedding-001",
    google_api_key=GEMINI_API_KEY,
)

# === STEP 3: Embed & Store in PGVector ===
vectorstore = PGVector.from_documents(
    documents=documents,
    embedding=embedding_model,
    connection=PGVECTOR_CONN_STRING,
    collection_name=COLLECTION_NAME,
    pre_delete_collection=True,  # Recreates the collection cleanly
)

print("✅ All document embeddings stored in PGVector.")


✅ Loaded 52 document chunks.
✅ All document embeddings stored in PGVector.
